# Morlet spectrograms: frequency and brain state

Run these five code cells in order; no other notebook section is needed.
This replaces the earlier spectrogram analyses with one calculation and one plotting function.

- The first figure has **brain states in rows and stimulation frequencies in columns**, at one amplitude and seed.
- The second compares **amplitudes and brain states at one stimulation frequency**.
- Overview pages retain **all other eligible runs**, including baselines, extra seeds and protocol variants.

Named **hex0/O1 comparison runs are excluded**. The `hex0` node set itself is retained because the other experiments also use it.
Main comparisons use connected, converged runs with 1.5 s onset, 1 s requested stimulation, 2.5 s recording and π/2 phase.
Timing, amplitude and random seed come from configuration files, not filename guesses. State follows the project's
`highCorrelation` naming convention. Missing conditions are left empty; individual runs are never seed-averaged.


In [1]:
from pathlib import Path
from types import SimpleNamespace
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink

morlet = SimpleNamespace(
    root=Path.cwd() if (Path.cwd() / "data.csv").exists() else Path.cwd().parent,
    node_sets=Path.home() / "Downloads/O1_data/node_sets.json",  # edit for your circuit
    frequencies=np.arange(1, 20.001, 0.125),
    periods=3.0,                # minimum cycles across the full wavelet window
    amplitude=1.0,             # first figure: selected V/m
    seed=843,                  # actual configuration seed
    stimulus_hz=10,            # second figure: selected stimulation frequency
    amplitudes=(0.25, 0.5, 1, 2, 4),
)
if str(morlet.root) not in sys.path:
    sys.path.insert(0, str(morlet.root))
from src.spectrogram_comparisons import load_runs, plot_grid
morlet.output = morlet.root / "output/morlet_spectrograms"
morlet.output.mkdir(parents=True, exist_ok=True)


C:\Users\Eloise\miniconda3\Lib\site-packages\h5py\__init__.py:36: UserWarning: h5py is running against HDF5 1.14.6 when it was built against 1.14.5, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


### How the spectrogram is calculated

**1. Turn spikes into a signal.** Count all population spikes in 5 ms bins and divide by the bin duration and total
configured neuron count, including silent neurons. The result $x(t)$ is firing rate in spikes/s/neuron. No extra smoothing is applied.

**2. Build a Morlet wavelet for each analyzed frequency.** A Morlet is a complex sinusoid inside a Gaussian envelope:

$$\psi_f(t) \propto [e^{i2\pi ft}-c]\,e^{-t^2/(2\sigma_f^2)},\qquad \sigma_f=\frac{N}{6f}.$$

Here $c$ makes the sampled wavelet sum to zero, so constant firing rate is rejected. Each wavelet is normalized to unit energy.
We retain approximately ±3σ, with both ends rounded outward to the 5 ms sample grid:

$$T(f)=2\Delta t\left\lceil\frac{N}{2f\Delta t}\right\rceil\geq\frac{N}{f},\qquad N\geq3.$$

| Analyzed frequency | Minimum window for N = 3 |
| --- | --- |
| 1 Hz | 3 s |
| 5 Hz | 0.6 s |
| 10 Hz | 0.3 s |
| 20 Hz | 0.15 s |

Thus higher frequencies use shorter windows. `periods` counts cycles across the **full support**, not the conventional
Morlet `n_cycles` parameter. For this envelope, conventional `n_cycles` would be $\pi N/3$.

**3. Slide the wavelet across the entire signal.** `fftconvolve` calculates the correlation with the reversed,
complex-conjugated wavelet. We keep a coefficient at every 5 ms sample, without restarting at stimulation boundaries:

$$P(f,t)=|x*\psi_f^*(-t)|^2.$$

This is the complete calculation in `morlet_power` in `src/spectrogram_comparisons.py`: build wavelet → reflect edges →
convolve → square magnitude. `morlet_kernel` in `src/spectrograms.py` constructs the wavelet.

**4. Plot power in dB.** Colour is $10\log_{10}(P)$, with a small numerical floor for zero power. Every figure uses
the same pooled 2nd–99th percentile limits. Dark patches mean low power. Smooth shading blends nearby dense samples
visually; it does not change the numerical transform. This is wavelet power, not a PSD or a significance map.

The signal is **reflected at the recording edges**, and all coefficients are displayed without cutouts or edge hatching.
Edge values depend on that reflection: a 2.5 s recording cannot contain a fully observed three-period window at 1 Hz.
Only genuinely unverified recording tails are shown in gray. Increasing `periods` improves frequency selectivity while
reducing time localization; a 0.125 Hz display grid is not 0.125 Hz resolving power.
See [MNE's Morlet explanation](https://mne.tools/stable/generated/mne.time_frequency.morlet.html) for the
relationship between Gaussian width, frequency and cycles; its support truncation convention differs.


In [2]:
morlet.runs, morlet.audit = load_runs(morlet.root, morlet.node_sets, morlet.frequencies, morlet.periods)
morlet.limits = np.percentile(np.concatenate([r["power_db"].ravel() for r in morlet.runs]), [2, 99])
if morlet.limits[0] == morlet.limits[1]:
    morlet.limits[1] += 1
morlet.inventory = pd.DataFrame([{key: r[key] for key in
    ("name", "state", "seed", "stimulus_hz", "amplitude_vm", "delay_s", "duration_s", "standard", "warnings")}
    for r in morlet.runs])
morlet.inventory.to_csv(morlet.output / "inventory.csv", index=False)
pd.DataFrame(morlet.audit).to_csv(morlet.output / "selection_audit.csv", index=False)
print(pd.DataFrame(morlet.audit).status.value_counts().to_string())
display(pd.DataFrame(morlet.audit).query("status != 'loaded' or reason != ''"))

def choose_run(state, frequency, amplitude):
    matches = [r for r in morlet.runs if r["standard"] and r["state"] == state
               and r["seed"] == morlet.seed and np.isclose(r["stimulus_hz"], frequency)
               and np.isclose(r["amplitude_vm"], amplitude)]
    if len(matches) > 1:
        raise ValueError(f"Multiple matching runs; select explicitly: {[r['name'] for r in matches]}")
    return matches[0] if matches else None


status
loaded      52
excluded     9
skipped      2


,name,status,reason
0,hex0_10Hz_0.25Vm,excluded,hex0/O1 comparison run
1,hex0_10Hz_0.25Vm_seed2,excluded,hex0/O1 comparison run
2,hex0_10Hz_1Vm,excluded,hex0/O1 comparison run
3,hex0_10Hz_4Vm,excluded,hex0/O1 comparison run
8,highCorrelation_sim10Hz0.5Vm_seed2,skipped,[Errno 2] Unable to synchronously open file (u...
9,highCorrelation_sim10Hz0.5Vm_seed3,skipped,[Errno 2] Unable to synchronously open file (u...
20,O1_10Hz_0.25Vm,excluded,hex0/O1 comparison run
21,O1_10Hz_1Vm_seed1,excluded,hex0/O1 comparison run
22,O1_10Hz_1Vm_seed2,excluded,hex0/O1 comparison run
23,O1_10Hz_1Vm_seed3,excluded,hex0/O1 comparison run


### Frequency × brain state

Each row is one brain state; each column is a stimulation frequency. The vertical axis inside each panel is the
**analyzed** frequency. All panels use the same amplitude, seed, time axes and colour limits.
Change `morlet.amplitude` or `morlet.seed` in the settings cell and rerun the section to select another comparison.


In [3]:
morlet.frequency_grid = [[choose_run(state, f, morlet.amplitude) for f in (7, 10, 15)] for state in ("low", "high")]
morlet.figure = plot_grid(morlet.frequency_grid, morlet.limits,
    f"Morlet power | {morlet.amplitude:g} V/m | seed {morlet.seed}",
    row_labels=["Low correlation", "High correlation"], column_labels=["7 Hz stimulation", "10 Hz stimulation", "15 Hz stimulation"])
morlet.figure.savefig(morlet.output / "frequency_and_state.png", dpi=140)
display(morlet.figure)
plt.close(morlet.figure)


<Figure size 1440x640 with 7 Axes>

### Amplitude × brain state at one stimulation frequency

Set `morlet.stimulus_hz` to 7, 10 or 15 Hz. Each row is an amplitude and each column is a brain state.


In [4]:
morlet.amplitude_grid = [[choose_run(state, morlet.stimulus_hz, amp) for state in ("low", "high")]
                         for amp in morlet.amplitudes]
# Omit rows missing in both states; preserve one-sided missing conditions.
morlet.available = [(amp, row) for amp, row in zip(morlet.amplitudes, morlet.amplitude_grid) if any(r is not None for r in row)]
morlet.figure = plot_grid([row for amp, row in morlet.available], morlet.limits,
    f"Morlet power | {morlet.stimulus_hz:g} Hz stimulation | seed {morlet.seed}",
    row_labels=[f"{amp:g} V/m" for amp, row in morlet.available], column_labels=["Low correlation", "High correlation"])
morlet.figure.savefig(morlet.output / "amplitude_and_state.png", dpi=140)
display(morlet.figure)
plt.close(morlet.figure)


<Figure size 960x1280 with 9 Axes>

### All other eligible runs

These saved overview pages include every eligible run, irrespective of the selected seed or amplitude:
baselines, additional seeds, long/delayed stimulation, replay, disconnected and unconverged runs.
They use simulation time and common axes so different durations remain visible. Check the inventory before comparing
different protocols. To inspect any run numerically, its linear power is `morlet.runs[i]["power"]`, with time and
frequency vectors in `"time"` and `"frequency"`. There is no separate normalization or baseline-ratio calculation.


In [5]:
morlet.overviews = []
for start in range(0, len(morlet.runs), 6):
    batch = morlet.runs[start:start+6]
    batch = batch + [None] * ((-len(batch)) % 3)
    rows = [batch[i:i+3] for i in range(0, len(batch), 3)]
    figure = plot_grid(rows, morlet.limits, f"All eligible recordings | page {start//6+1}", align_onset=False)
    filename = morlet.output / f"all_runs_{start//6+1:02d}.png"
    figure.savefig(filename, dpi=140)
    plt.close(figure)
    morlet.overviews.append(filename)
for filename in [morlet.output / "frequency_and_state.png", morlet.output / "amplitude_and_state.png", *morlet.overviews]:
    display(FileLink(Path(os.path.relpath(filename, Path.cwd())).as_posix()))
print(f"Saved {len(morlet.overviews)+2} figures to {morlet.output}")


Saved 11 figures to D:\work\ti-coupling-project\output\morlet_spectrograms


D:\work\ti-coupling-project\output\morlet_spectrograms\frequency_and_state.png

D:\work\ti-coupling-project\output\morlet_spectrograms\amplitude_and_state.png

D:\work\ti-coupling-project\output\morlet_spectrograms\all_runs_01.png

D:\work\ti-coupling-project\output\morlet_spectrograms\all_runs_02.png

D:\work\ti-coupling-project\output\morlet_spectrograms\all_runs_03.png

... (2 outputs left out)

D:\work\ti-coupling-project\output\morlet_spectrograms\all_runs_06.png

D:\work\ti-coupling-project\output\morlet_spectrograms\all_runs_07.png

D:\work\ti-coupling-project\output\morlet_spectrograms\all_runs_08.png

D:\work\ti-coupling-project\output\morlet_spectrograms\all_runs_09.png